In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from pathlib import Path

import polars as pl
from IPython.display import display
from polars.testing import assert_frame_equal
from tabseq_torch.dl.encoder import (
    Cat2Code,
    CatNum,
    DataCatalog,
    Encoder,
    cat2code_size,
    encode_cat,
    filter_part,
    make_cat2code,
    make_catnum,
    write_json,
)
from tabseq_torch.dl.lazy_dataset import (
    CatMetadata,
    ColumnType,
    LazyDataset,
    NumMetadata,
    TargetsMetadata,
)


In [ ]:
pl.__version__ == "1.37.1"

# Profiling

In [ ]:
tx_path = Path(
    "/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest"
)
print(tx_path.exists())
labels_path = tx_path.parent / "flag.parquet"
q_labels = pl.scan_parquet(labels_path)
print(q_labels.collect().estimated_size(unit="mb"))
display(q_labels.collect_schema())
display(q_labels.select(pl.col("flag").n_unique()).collect().item())
q_tx = pl.scan_parquet(tx_path)
display(q_tx.select("mcc").unique().collect().sort(by="mcc"))

# Initiate data catalog

In [ ]:
dc = DataCatalog(
    src_tabts_path=tx_path,
    cat_path=tx_path.parent / "cat",
    meta_path=tx_path.parent / "meta",
)

# Encoding flow

encode all categprical features and 

## columns profiling

In [ ]:
setname = "allcat"
q = pl.scan_parquet(dc.src)
display(q.collect_schema())
num_columns = ["amnt", "days_before", "transaction_number", "hour_diff"]
display(q_tx.select(pl.col("hour_diff").n_unique()).collect())
col_filter = num_columns + ["__index_level_0__", "app_id"]
cat_columns = [c for c in q.collect_schema() if c not in col_filter]
print("c=n_uniq:")
for c in cat_columns:
    print(f"{c}={q_tx.select(pl.col(c).n_unique()).collect().item()}")

## Create and store catnum by cat2code

In [ ]:
encode_cat(
    data_path=dc.src,
    dc=dc,
    columns=cat_columns,
    groupby="app_id",
    setname=setname,
)

# Encoded data validation

In [ ]:
def get_part_for_test(lf: pl.LazyFrame) -> pl.LazyFrame:
    return filter_part(
        lf=lf.select(["app_id", "mcc"]),
        part_col="app_id",
        num_parts=100,
        part=0,
    )

In [ ]:
decoded_part = get_part_for_test(pl.scan_parquet(dc.cat(setname=setname)))
expected_part = get_part_for_test(q_tx)
display(decoded_part.head().collect())
display(expected_part.head().collect())
assert_frame_equal(decoded_part, expected_part)